# Atera Pre-process with H&E

Turn a 10x Genomics **Atera WTA** output bundle into Celldega DegaFiles, and add the post-run **H&E image** as an aligned background. The DegaFiles used in the [Brain Hippocampus](../../tutorial_notebooks/Atera_Brain_Hippocampus_Landscape-Matrix/), [Breast Cancer FF](../../tutorial_notebooks/Atera_Breast_Cancer_Landscape-Matrix/) and [Colorectal Cancer](../../tutorial_notebooks/Atera_Colorectal_Cancer_Landscape-Matrix/) H&E tutorials were made this way.

Steps:
1. Download the bundle, H&E image and H&E alignment matrix
2. Extract the files Celldega needs
3. Pre-process into DegaFiles (optionally with parquet **row groups**)
4. Convert the H&E image so libvips can read it
5. Add the aligned H&E image
6. View the result in a Landscape

**Requirements:** `pip install "celldega[pre]>=0.29"` and a working [libvips](https://www.libvips.org/install.html) installation. `tifffile`, used to convert the H&E image, is installed with Celldega.

**Disk space:** Atera bundles are large. The hippocampus `_outs.zip` is 49 GB (colorectal 75 GB, breast 89 GB), and its extracted files take about 28 GB more, before the DegaFiles are written. See [Atera Pre-process](../Atera_Pre-process/) for the run time and memory of a large region.

## 1. Download

Datasets used in the Celldega documentation (10x Genomics Atera, Atera Onboard Analysis 1.1.1):

| Tissue | Sample name | 10x dataset page | DegaFiles in the docs |
|---|---|---|---|
| Human brain hippocampus, FF | `Atera_v1_WTA_S4A_Brain_Hippocampus_ND_FF` | [Fresh Frozen Human Brain](https://www.10xgenomics.com/datasets/atera-wta-ff-human-nd-brain) | [`celldega_atera-wta-ff-human-nd-brain`](https://github.com/broadinstitute/celldega_atera-wta-ff-human-nd-brain) |
| Human breast cancer, FF | `Atera_v1_WTA_S4C_Breast_Cancer_FF` | [Fresh Frozen Human Breast Cancer](https://www.10xgenomics.com/datasets/atera-wta-ff-human-breast-cancer) | [`celldega_atera-wta-ff-human-breast-cancer`](https://github.com/broadinstitute/celldega_atera-wta-ff-human-breast-cancer) |
| Human colorectal cancer, FF | `Atera_v1_WTA_S4B_Colorectal_Cancer_FF` | [Fresh Frozen Human Colorectal Cancer](https://www.10xgenomics.com/datasets/atera-wta-ff-human-crc) | [`celldega_atera-wta-ff-human-crc`](https://github.com/broadinstitute/celldega_atera-wta-ff-human-crc) |

The original data are licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

Each sample has four downloads on its 10x Genomics dataset page. Which ones Celldega needs:

| File | Needed? | Purpose |
|---|---|---|
| `<sample>_outs.zip` | **yes** | the Atera output bundle (cells, transcripts, cell-by-gene matrix, 2D morphology images) |
| `<sample>_he_image.ome.tif` | **yes** | the **post-run H&E image** (listed on the dataset page as *Supplemental: Post-Run H&E Image (OME-TIFF)*). This is the image shown as the H&E background. |
| `<sample>_he_imagealignment.csv` | **yes** | the **H&E alignment**: a headerless 3x3 affine matrix mapping H&E pixels to the reference (morphology) image pixels |
| `<sample>_keypoints.csv` | no | the matched keypoints used by 10x to compute the alignment; informational only, Celldega does not read it |

Use the `_he_image.ome.tif` that belongs to the same sample as the bundle, together with that sample's own `_he_imagealignment.csv`; each alignment matrix is specific to its image.

In [ ]:
sample = "Atera_v1_WTA_S4A_Brain_Hippocampus_ND_FF"  # change to process another dataset

data_root = "data/atera"             # bundle is extracted to data_root/sample
download_dir = f"data/atera_dl/{sample}"

url_bundle = f"https://s3-us-west-2.amazonaws.com/10x.files/samples/atera/1.1.1/{sample}/{sample}_outs.zip"
url_cf = f"https://cf.10xgenomics.com/samples/atera/1.1.1/{sample}/{sample}"

he_image = f"{download_dir}/{sample}_he_image.ome.tif"
he_alignment = f"{download_dir}/{sample}_he_imagealignment.csv"

In [ ]:
%%bash -s "$download_dir" "$url_bundle" "$url_cf"
# -C - resumes interrupted downloads (the bundles are large)
mkdir -p "$1" && cd "$1"
curl -sSf -C - -O "$2"
curl -sSf -C - -O "$3_he_image.ome.tif"
curl -sSf -C - -O "$3_he_imagealignment.csv"

## 2. Extract

Celldega reads `cells.zarr.zip`, `transcripts.zarr.zip`, `cell_feature_matrix.zarr.zip` directly (leave them zipped) and the 2D morphology images in `morphology_2d/`. The 3D morphology stack (`morphology_3d/`, 21 GB for the hippocampus) is not used and is skipped below. The bundle must end up in `data_root/sample`.

In [ ]:
%%bash -s "$download_dir" "$data_root" "$sample"
mkdir -p "$2/$3"
unzip -q -o "$1/$3_outs.zip" -x 'morphology_3d/*' -d "$2/$3"
# optional: free up space once extracted
# rm "$1/$3_outs.zip"

## 3. Pre-process into DegaFiles

`dega.pre.main` detects the Atera bundle and writes the DegaFiles (image pyramids, transcript tiles, cell boundary tiles, gene matrices, ...).

Choose one of two layouts:

- `use_row_groups=False` (default): one parquet file per tile. Works well on static hosts such as **GitHub Pages**.
- `use_row_groups=True`: tiles are stored as row groups in a small number of chunked parquet files (`max_row_groups_per_file` per file). Fewer files, suited to hosting such as **Hugging Face**.

In [ ]:
import celldega as dega

use_row_groups = False  # True -> row-group DegaFiles
suffix = "row_groups" if use_row_groups else "tiles"
path_dega_files = f"data/dega_files/{suffix}/{sample}"

dega.pre.main(
    sample=sample,
    data_root_dir=data_root,
    tile_size=250,
    image_tile_layer="all",
    path_dega_files=path_dega_files,
    max_workers=8,                  # parallel tile generation; adjust to your machine
    use_row_groups=use_row_groups,
    max_row_groups_per_file=400,    # only used when use_row_groups=True
)

## 4. Convert the H&E image for libvips

The Atera H&E OME-TIFF stores its red, green and blue values as three **separate planes** (planar configuration 2, shape `3 x height x width`) with a built-in image pyramid. libvips, which Celldega uses to build the H&E tile pyramid, cannot read this layout and fails with

```
tiff2vips: tiled separate planes not supported
```

So convert the full-resolution image once to a standard **interleaved** (`height x width x 3`) tiled RGB TIFF. This does not change the pixels or their coordinates, so the alignment CSV still applies. Reading the image needs enough RAM for the uncompressed image: the hippocampus image is 3 x 45,694 x 34,152 pixels, about 4.7 GB, and the conversion needs about twice that.

In [ ]:
import numpy as np
import tifffile

he_interleaved = f"{download_dir}/{sample}_he_interleaved.tif"

with tifffile.TiffFile(he_image) as tif:
    page = tif.pages[0]  # full-resolution image
    assert page.planarconfig == 2 and page.shape[0] == 3 and page.dtype == np.uint8, (
        page.shape, page.dtype, page.planarconfig
    )
    rgb = page.asarray()  # (3, H, W)

rgb = np.ascontiguousarray(np.moveaxis(rgb, 0, -1))  # (H, W, 3)
tifffile.imwrite(
    he_interleaved, rgb, photometric="rgb", tile=(512, 512),
    compression="zlib", predictor=True, bigtiff=True,
)
del rgb

## 5. Add the aligned H&E image

[`dega.pre.add_aligned_he`](../../../python/aligned_he/) resamples the H&E image onto the reference (fluorescence) image canvas and adds a WebP DeepZoom pyramid to an **existing** DegaFiles folder. A **Fluorescence / H&E** selector then appears in the viewer.

- Pass the **converted** H&E image from step 4 (the interleaved TIFF), not the original planar OME-TIFF.
- `alignment` is the sample's `_he_imagealignment.csv`: a headerless 3x3 affine matrix mapping H&E pixels to reference-image pixels.
- The H&E image must be 8-bit RGB or RGBA.
- An existing H&E layer is not overwritten.
- Run this for each DegaFiles folder you create (for example both the tile and row-group versions).

In [ ]:
dega.pre.add_aligned_he(
    path_dega_files,
    he_interleaved,            # converted image from step 4
    alignment=he_alignment,    # <sample>_he_imagealignment.csv
)

## 6. View

Serve the DegaFiles folder locally and open it in a Landscape. Reload the Landscape after adding the H&E image to see the Fluorescence / H&E selector.

In [ ]:
dega.viz.Landscape(
    technology="Xenium",
    base_url=f"http://localhost:{dega.viz.get_local_server()}/{path_dega_files}",
)

## Hosting

- **GitHub:** push the one-file-per-tile DegaFiles folder to a repository. Large folders need several commits; [`scripts/upload_degafiles_batched.py`](https://github.com/broadinstitute/celldega/blob/main/scripts/upload_degafiles_batched.py) in the Celldega repository pushes an existing DegaFiles Git repository in bounded, resumable batches.
- **Hugging Face:** upload the row-group DegaFiles folder to a dataset repository.

Point `base_url` at the hosted folder to share the Landscape, for example `base_url="https://raw.githubusercontent.com/<org>/<repo>/main"`.